# Week 62, Day 3: Markov random fields, the Ising model and maximum entropy

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w062/d3.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Independence has the most entropy

For three binary units with means 0.2, 0.5 and 0.7 (probability of being on), build the independent distribution over the 8 states. Build correlated distributions with the same means by moving probability mass along a direction that leaves all three means unchanged, and check that each has lower entropy.

In [ ]:
import numpy as np
import itertools
states = np.array(list(itertools.product((0, 1), repeat=3)))
m = np.array([0.2, 0.5, 0.7])
p_ind = ...                                            # product of Bernoulli probabilities, one per state

def means(p):
    return states.T @ p

def entropy(p):
    p = p[p > 0]
    return -(p * np.log(p)).sum()

# a direction in the space of distributions that leaves total mass and all three means unchanged
d = np.array([1, -1, -1, 1, -1, 1, 1, -1], dtype=float)
p_others = [p_ind + t * d for t in (-0.02, 0.01, 0.02, 0.025)]
p_corr = p_others[2]
print("means:", means(p_ind), means(p_corr))
print(f"entropy: independent {entropy(p_ind):.4f}; correlated", [round(float(entropy(q)), 4) for q in p_others])

In [ ]:
import numpy as np
def test_same_means():
    "the correlated distribution has the same means as the independent one"
    assert np.allclose(means(p_corr), means(p_ind))
def test_entropy():
    "the independent distribution has higher entropy than any correlated one with the same means"
    assert all(entropy(p_ind) > entropy(q) for q in p_others)

test_same_means()
test_entropy()
print("All checks passed")

### Exercise: An Ising model of eight neurons

Enumerate all 256 states of 8 units with values -1 and +1, compute the Ising energy with biases h and couplings J (a symmetric matrix with zero diagonal), the probabilities, and from them the mean of each unit and the correlation matrix.

In [ ]:
S8 = np.array(list(itertools.product((-1, 1), repeat=8)))
h = np.full(8, -0.8)                                   # neurons are mostly silent
J = np.zeros((8, 8)); J[0, 1] = J[1, 0] = 0.6; J[2, 3] = J[3, 2] = -0.4
E = -(S8 @ h) - 0.5 * np.einsum("ki,ij,kj->k", S8, J, S8)
P8 = ...
mean8 = S8.T @ P8
cov8 = (S8 * P8[:, None]).T @ S8 - np.outer(mean8, mean8)
C8 = cov8 / np.sqrt(np.outer(np.diag(cov8), np.diag(cov8)))
print("mean of each unit:", np.round(mean8, 3))
print(f"correlation 0-1 (coupled +0.6) {C8[0, 1]:.3f}; 2-3 (coupled -0.4) {C8[2, 3]:.3f}; 0-7 (uncoupled) {C8[0, 7]:.3f}")

In [ ]:
import numpy as np
def test_norm():
    "probabilities over all 256 states sum to one"
    assert abs(P8.sum() - 1) < 1e-12
def test_effect():
    "a positive coupling between neurons 0 and 1 raises their correlation above that of an uncoupled pair"
    assert C8[0, 1] > C8[0, 7] + 0.1

test_norm()
test_effect()
print("All checks passed")

### Exercise: Gibbs sampling a lattice

Complete the checkerboard Gibbs sampler: update all 'black' squares at once given their neighbours, then all 'white' ones. Run 300 sweeps on a 40 by 40 lattice for couplings 0.2, 0.44 and 0.7, starting from random states, and record the absolute mean state over the last 100 sweeps.

In [ ]:
def gibbs_lattice(J_, n_side, sweeps, g):
    s = g.choice([-1, 1], size=(n_side, n_side))
    colour = (np.add.outer(np.arange(n_side), np.arange(n_side)) % 2).astype(bool)
    trace = []
    for _ in range(sweeps):
        for c in (colour, ~colour):
            nb = np.roll(s, 1, 0) + np.roll(s, -1, 0) + np.roll(s, 1, 1) + np.roll(s, -1, 1)
            p_up = ...
            new = np.where(g.random(s.shape) < p_up, 1, -1)
            s = np.where(c, new, s)
        trace.append(abs(s.mean()))
    return s, np.array(trace)

gl = np.random.default_rng(623)
mag = {}
for J_ in (0.2, 0.44, 0.7):
    _, tr = gibbs_lattice(J_, 40, 300, gl)
    mag[J_] = tr[-100:].mean()
print({k: round(float(v), 3) for k, v in mag.items()})

In [ ]:
import numpy as np
def test_transition():
    "magnetisation is small below the critical coupling and large above it"
    assert mag[0.2] < 0.2 and mag[0.7] > 0.8

test_transition()
print("All checks passed")